# Northstar Incremental Employee Merge

This notebook demonstrates an incremental processing pattern for employee data using a persisted watermark and Delta Lake MERGE semantics.

The workflow simulates a subsequent source arrival containing both new and changed employee records, processes only records newer than the stored watermark, and applies inserts and updates to the Silver employee Delta table without rebuilding the full dataset.

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
# Northstar incremental processing configuration

silver_employee_path = "abfss://silver@stnenimadlsdev01.dfs.core.windows.net/northstar/employees"
watermark_path = "abfss://silver@stnenimadlsdev01.dfs.core.windows.net/northstar/control/employee_watermark"

print(f"Silver employee target: {silver_employee_path}")
print(f"Watermark location: {watermark_path}")

In [0]:
# Load the persisted watermark.
# If this is the first incremental run, start from the baseline timestamp.

default_watermark = "1900-01-01 00:00:00"

try:
    watermark_df = spark.read.format("delta").load(watermark_path)

    last_watermark = (
        watermark_df
        .agg(F.max("watermark_timestamp").alias("last_watermark"))
        .first()["last_watermark"]
    )

    if last_watermark is None:
        last_watermark = default_watermark

except Exception:
    last_watermark = default_watermark

print(f"Current watermark: {last_watermark}")

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    TimestampType
)

# Simulate a subsequent employee source arrival.
# Existing employee IDs represent updates; new IDs represent inserts.

incremental_data = [
    (
        "EE0000001", "EMP1001", "Samuel", "Campbell",
        "1990-05-14", "2018-02-01", None,
        "Active", "Information Technology", "VA",
        "2026-08-18 14:00:00"
    ),
    (
        "EE0000002", "EMP1002", "Ethan", "Brown",
        "1988-11-02", "2017-06-15", None,
        "Active", "Finance", "MD",
        "2026-08-18 14:05:00"
    ),
    (
        "EE0010001", "EMP2001", "Taylor", "Morgan",
        "1993-03-21", "2026-08-18", None,
        "Active", "Customer Operations", "VA",
        "2026-08-18 14:10:00"
    ),
    (
        "EE0010002", "EMP2002", "Cameron", "Davis",
        "1991-09-09", "2026-08-18", None,
        "Active", "Human Resources", "NC",
        "2026-08-18 14:15:00"
    )
]

incremental_schema = StructType([
    StructField("employee_id", StringType(), False),
    StructField("employer_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("date_of_birth", StringType(), True),
    StructField("hire_date", StringType(), True),
    StructField("termination_date", StringType(), True),
    StructField("employment_status", StringType(), True),
    StructField("department", StringType(), True),
    StructField("state", StringType(), True),
    StructField("source_updated_at", StringType(), True)
])

incoming_df = (
    spark.createDataFrame(incremental_data, schema=incremental_schema)
    .withColumn("date_of_birth", F.to_date("date_of_birth"))
    .withColumn("hire_date", F.to_date("hire_date"))
    .withColumn("termination_date", F.to_date("termination_date"))
    .withColumn("source_updated_at", F.to_timestamp("source_updated_at"))
)

incoming_df.show(truncate=False)

In [0]:
# Filter the incoming batch to records newer than the persisted watermark.

incremental_df = incoming_df.filter(
    F.col("source_updated_at") > F.to_timestamp(F.lit(str(last_watermark)))
)

incremental_count = incremental_df.count()

print(f"Records eligible for incremental processing: {incremental_count}")

incremental_df.orderBy("source_updated_at").show(truncate=False)

In [0]:
merge_df = (
    incremental_df
    .withColumn("processing_timestamp", F.current_timestamp())
    .withColumn("source_system", F.lit("Northstar Incremental Demo"))
    .withColumn("source_file", F.lit("incremental_employee_batch_20260818"))
    .withColumn("_corrupt_record", F.lit(None).cast("string"))
    .withColumn("batch_id", F.lit("INC-20260818"))
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("employee_id"),
                F.col("employer_id"),
                F.col("first_name"),
                F.col("last_name"),
                F.col("employment_status"),
                F.col("department"),
                F.col("state")
            ),
            256
        )
    )
    .withColumn(
        "employee_age",
        F.floor(F.months_between(F.current_date(), F.col("date_of_birth")) / 12)
    )
    .withColumn(
        "years_of_service",
        F.round(F.months_between(F.current_date(), F.col("hire_date")) / 12, 2)
    )
    .withColumn(
        "coverage_eligible_as_of_date",
        F.when(
            (F.col("employment_status") == "Active") &
            (F.col("hire_date") <= F.current_date()),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
)

In [ ]:
# ---------------------------------------------------------
# Apply the incremental batch to Silver via Delta MERGE.
# whenMatchedUpdateAll handles existing employee_id rows (updates);
# whenNotMatchedInsertAll handles new employee_id rows (inserts).
# ---------------------------------------------------------

merge_columns = [
    "employee_id",
    "employer_id",
    "first_name",
    "last_name",
    "date_of_birth",
    "hire_date",
    "termination_date",
    "employment_status",
    "department",
    "state",
    "batch_id",
    "source_system",
    "source_file",
    "processing_timestamp",
    "record_hash",
    "employee_age",
    "years_of_service",
    "coverage_eligible_as_of_date",
    "_corrupt_record",
]

merge_source_df = merge_df.select(*merge_columns)

if incremental_count > 0:
    target_table = DeltaTable.forPath(spark, silver_employee_path)

    (
        target_table.alias("target")
        .merge(
            merge_source_df.alias("source"),
            "target.employee_id = source.employee_id",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"MERGE applied: {incremental_count} records processed (updates + inserts).")
else:
    print("No incremental records to merge; skipping MERGE.")

In [0]:
# Advance and persist the watermark after a successful MERGE.

new_watermark = (
    incremental_df
    .agg(F.max("source_updated_at").alias("watermark_timestamp"))
    .first()["watermark_timestamp"]
)

if new_watermark is not None:
    watermark_output_df = spark.createDataFrame(
        [(new_watermark,)],
        ["watermark_timestamp"]
    )

    (
        watermark_output_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(watermark_path)
    )

    print(f"Watermark advanced to: {new_watermark}")
else:
    print("No incremental records processed; watermark unchanged.")

In [0]:
# Validate the incremental result.

validation_ids = [
    "EE0000001",
    "EE0000002",
    "EE0010001",
    "EE0010002"
]

final_silver_df = spark.read.format("delta").load(silver_employee_path)

print(f"Silver employee row count: {final_silver_df.count()}")

final_silver_df.filter(
    F.col("employee_id").isin(validation_ids)
).select(
    "employee_id",
    "employer_id",
    "first_name",
    "last_name",
    "employment_status",
    "department",
    "state",
    "batch_id",
    "source_system",
    "processing_timestamp"
).orderBy("employee_id").show(truncate=False)

persisted_watermark = (
    spark.read.format("delta")
    .load(watermark_path)
    .agg(F.max("watermark_timestamp").alias("watermark_timestamp"))
    .first()["watermark_timestamp"]
)

print(f"Persisted watermark: {persisted_watermark}")